# 👹 AN2DL Challenge 2: The Grumpy Doctogres Challenge

## Notebook 01 B: Post Preprocessing

In [1]:
import math
import os
from concurrent.futures import ThreadPoolExecutor, Future

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from PIL.ImageFile import ImageFile
from sklearn.model_selection import StratifiedKFold
from torchvision import transforms
from tqdm import tqdm

from internal.persistence_manager import PersistenceManager

# check if it is cuda available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
cuda_is_available = torch.cuda.is_available()
data = PersistenceManager.load_dataset()

Using device: cuda
Arrays and scalers loaded successfully from: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib


In [2]:
IMAGE_SIZE = 640

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

# RandomResizedCrop is often better than plain Resize for generalization
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.8, 1.0),  # don’t go too low or we lose tumor context
        ratio=(0.9, 1.1),  # keep aspect close to square
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15, fill=(255, 255, 255)),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])


val_test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


In [3]:
K = 5

train_df = data.train_df.sample(frac=1.0, random_state=42).reset_index(drop=True)  # shuffle once
train_df["fold"] = -1

skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label_idx"])):
    train_df.loc[val_idx, "fold"] = fold

train_df["fold"].value_counts()

# now each row has a `fold` \in {0,1,2,3,4}

fold
0    117
2    116
1    116
3    116
4    116
Name: count, dtype: int64

In [4]:
train_df.head()

,sample_index,image_path,mask_path,label,label_idx,width,height,mask_pixels,bbox_area,fold
0,img_0194,../data/train_data/img_0194.png,../data/train_data/mask_0194.png,Luminal B,0,1514,1024,23147,320276,0
1,img_0306,../data/train_data/img_0306.png,../data/train_data/mask_0306.png,Luminal A,1,1198,1024,10704,157192,0
2,img_0619,../data/train_data/img_0619.png,../data/train_data/mask_0619.png,Luminal B,0,1024,1680,8728,264550,0
3,img_0514,../data/train_data/img_0514.png,../data/train_data/mask_0514.png,Luminal B,0,1024,1141,32460,401208,2
4,img_0308,../data/train_data/img_0308.png,../data/train_data/mask_0308.png,HER2(+),2,1024,1921,18391,323080,1


### Save Processed DataFrames

This section saves the processed training and test DataFrames using the `PersistenceManager` for future use in model training and evaluation.

In [5]:
PersistenceManager.save_dataset({
    "train_df": train_df,
    "test_df": data.test_df,
    "train_transforms": train_transforms,
    "val_test_transforms": val_test_transforms,
    "idx2label": data.idx2label,
    "label2idx": data.label2idx,
    "num_K_folds": 5,
    "image_size": IMAGE_SIZE
})

Arrays and scalers saved successfully to: /home/andre/university/AN2DL-Challenge-2/notebooks/processed/dataset.joblib
